# emg2qwerty — BiGRU Runner

This notebook mirrors the existing baseline runner workflow, but switches the training target to the new **BiGRU-CTC** model (`model=bigru_ctc`).

It runs:

`python -c "<patch torch.load; runpy.run_module('emg2qwerty.train', run_name='__main__')>" user=single_user model=bigru_ctc ...`

It also keeps the same Windows-friendly fixes:
- patches `torch.load(weights_only=False)` inside the training process
- keeps the optional KenLM import patch in `decoder.py`
- uses `num_workers=1` to stay compatible with the upstream dataloader settings on Windows


In [1]:
from __future__ import annotations
import os, sys, subprocess, time, re
from pathlib import Path

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "emg2qwerty").exists() and (REPO_ROOT / "config").exists(),     f"Open this notebook from the emg2qwerty repo root. CWD={REPO_ROOT}"

print("Python:", sys.version.split()[0])
print("Repo root:", REPO_ROOT)


Python: 3.10.20
Repo root: c:\Users\15855\OneDrive\UCLA\course\247A\Project\emg2qwerty


## 1) GPU sanity check

In [2]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available in this kernel. Select the correct kernel / reinstall CUDA torch.")

print("gpu:", torch.cuda.get_device_name(0))
print("torch CUDA:", torch.version.cuda)


torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA GeForce RTX 5080
torch CUDA: 12.8


## 2) DATA_DIR (folder containing `*.hdf5`)

In [3]:
DEFAULT_IN_REPO = REPO_ROOT / "data" / "subject_89335547"

# If your data is elsewhere, set this to the folder that directly contains *.hdf5
# EXTERNAL_DATA_DIR = r"C:\Users\...\subject_89335547"
EXTERNAL_DATA_DIR = None

data_dir = DEFAULT_IN_REPO if DEFAULT_IN_REPO.exists() else (Path(EXTERNAL_DATA_DIR) if EXTERNAL_DATA_DIR else DEFAULT_IN_REPO)
data_dir = data_dir.resolve()
print("DATA_DIR:", data_dir)

hdf5 = sorted(data_dir.glob("*.hdf5"))
print("Found .hdf5 files:", len(hdf5))
if len(hdf5) == 0:
    raise FileNotFoundError(
        f"No .hdf5 files found in {data_dir}\n"
        f"Put files at: {DEFAULT_IN_REPO} (recommended) OR set EXTERNAL_DATA_DIR."
    )

DATA_DIR_HYDRA = str(data_dir).replace("\\", "/")
print("DATA_DIR_HYDRA:", DATA_DIR_HYDRA)
print("Example files:", [p.name for p in hdf5[:5]])


DATA_DIR: C:\Users\15855\OneDrive\UCLA\course\247A\Project\emg2qwerty\data\subject_89335547
Found .hdf5 files: 18
DATA_DIR_HYDRA: C:/Users/15855/OneDrive/UCLA/course/247A/Project/emg2qwerty/data/subject_89335547
Example files: ['2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-02-1622682789-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-03-1622764398-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5']


## 3) Windows fix: make KenLM optional (greedy decoder doesn't need it)

In [4]:
import importlib

decoder_py = REPO_ROOT / "emg2qwerty" / "decoder.py"
assert decoder_py.exists(), f"Missing file: {decoder_py}"

def patch_decoder_make_kenlm_optional(path: Path) -> bool:
    txt = path.read_text(encoding="utf-8", errors="ignore")
    if "except ImportError" in txt and "kenlm = None" in txt:
        print("decoder.py already patched (kenlm optional).")
        return False

    pattern = r"(^|\n)\s*import\s+kenlm\s*(\n|$)"
    if re.search(pattern, txt):
        repl = r"\1try:\n    import kenlm  # type: ignore\nexcept ImportError:\n    kenlm = None  # type: ignore\n\2"
        txt2 = re.sub(pattern, repl, txt, count=1)
    elif "import kenlm" in txt:
        txt2 = txt.replace("import kenlm", "try:\n    import kenlm  # type: ignore\nexcept ImportError:\n    kenlm = None  # type: ignore", 1)
    else:
        print("No 'import kenlm' found; no patch needed.")
        return False

    path.write_text(txt2, encoding="utf-8", newline="\n")
    print("Patched decoder.py to make kenlm optional.")
    return True

try:
    import emg2qwerty.decoder  # noqa: F401
    print("Import emg2qwerty.decoder: OK")
except Exception as e:
    msg = repr(e)
    print("Import emg2qwerty.decoder failed:", msg)
    if "kenlm" in msg.lower():
        changed = patch_decoder_make_kenlm_optional(decoder_py)
        importlib.invalidate_caches()
        import emg2qwerty.decoder  # noqa: F401
        print("Re-import after patch: OK")
    else:
        raise


Import emg2qwerty.decoder: OK


## 4) 1-epoch smoke test training for BiGRU-CTC

In [5]:
def run_and_print(cmd: list[str], cwd: Path) -> None:
    env = os.environ.copy()
    env["HYDRA_FULL_ERROR"] = "1"
    print(">>", " ".join(cmd))
    p = subprocess.run(cmd, cwd=str(cwd), env=env, text=True, capture_output=True)
    if p.stdout:
        print("----- STDOUT (tail) -----")
        print(p.stdout[-8000:])
    if p.stderr:
        print("----- STDERR (tail) -----")
        print(p.stderr[-8000:])
    if p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}. See STDOUT/STDERR above.")

NUM_WORKERS = 1  # keep >0 because upstream uses persistent_workers=True

patch_and_run = r'''
import torch, runpy
_old = torch.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _old(*args, **kwargs)
torch.load = _patched_load
runpy.run_module("emg2qwerty.train", run_name="__main__")
'''.strip()

cmd = [
    sys.executable, "-c", patch_and_run,
    "user=single_user",
    "model=bigru_ctc",
    f"dataset.root={DATA_DIR_HYDRA}",
    "trainer.accelerator=gpu", "trainer.devices=1",
    "trainer.max_epochs=1",
    f"num_workers={NUM_WORKERS}",
    "train=True",
    "decoder=ctc_greedy",
]

t0 = time.time()
run_and_print(cmd, cwd=REPO_ROOT)
print(f"Smoke test done in {(time.time()-t0)/60:.1f} min")


>> c:\Users\15855\.conda\envs\emg2qwerty\python.exe -c import torch, runpy
_old = torch.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _old(*args, **kwargs)
torch.load = _patched_load
runpy.run_module("emg2qwerty.train", run_name="__main__") user=single_user model=bigru_ctc dataset.root=C:/Users/15855/OneDrive/UCLA/course/247A/Project/emg2qwerty/data/subject_89335547 trainer.accelerator=gpu trainer.devices=1 trainer.max_epochs=1 num_workers=1 train=True decoder=ctc_greedy
----- STDOUT (tail) -----
▌  | 96/127 [00:31<00:10,  3.01it/s, loss=165, v_num=0]
Epoch 0:  94%|█████████▍| 120/127 [00:39<00:02,  3.05it/s, loss=170, v_num=0]

Validation: 0it [00:00, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 0:  95%|█████████▌| 121/127 [00:39<00:01,  3.05it/s, loss=170, v_num=0]

Epoch 0:  96%|█████████▌| 122/127 [00:39<00:01,  3.06it/s, loss=170, v_num=0]

Epoch 0

## 5) Full BiGRU-CTC training run

In [6]:
EPOCHS = 40
NUM_WORKERS = 1

patch_and_run = r'''
import torch, runpy
_old = torch.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _old(*args, **kwargs)
torch.load = _patched_load
runpy.run_module("emg2qwerty.train", run_name="__main__")
'''.strip()

cmd = [
    sys.executable, "-c", patch_and_run,
    "user=single_user",
    "model=bigru_ctc",
    f"dataset.root={DATA_DIR_HYDRA}",
    "trainer.accelerator=gpu", "trainer.devices=1",
    f"trainer.max_epochs={EPOCHS}",
    f"num_workers={NUM_WORKERS}",
    "train=True",
    "decoder=ctc_greedy",
]

print("Ready. Full BiGRU command:")
print(" ".join(cmd))
run_and_print(cmd, cwd=REPO_ROOT)


Ready. Full BiGRU command:
c:\Users\15855\.conda\envs\emg2qwerty\python.exe -c import torch, runpy
_old = torch.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _old(*args, **kwargs)
torch.load = _patched_load
runpy.run_module("emg2qwerty.train", run_name="__main__") user=single_user model=bigru_ctc dataset.root=C:/Users/15855/OneDrive/UCLA/course/247A/Project/emg2qwerty/data/subject_89335547 trainer.accelerator=gpu trainer.devices=1 trainer.max_epochs=40 num_workers=1 train=True decoder=ctc_greedy
>> c:\Users\15855\.conda\envs\emg2qwerty\python.exe -c import torch, runpy
_old = torch.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _old(*args, **kwargs)
torch.load = _patched_load
runpy.run_module("emg2qwerty.train", run_name="__main__") user=single_user model=bigru_ctc dataset.root=C:/Users/15855/OneDrive/UCLA/course/247A/Project/emg2qwerty/data/subject_89335547 trainer.accelerator=gpu trai

## 6) Optional: freeze the latest run into `results/`

In [7]:
freeze_cmd = [sys.executable, "scripts/freeze_run.py"]
print("Freeze command:")
print(" ".join(freeze_cmd))
run_and_print(freeze_cmd, cwd=REPO_ROOT)


Freeze command:
c:\Users\15855\.conda\envs\emg2qwerty\python.exe scripts/freeze_run.py
>> c:\Users\15855\.conda\envs\emg2qwerty\python.exe scripts/freeze_run.py
----- STDOUT (tail) -----
Freezing run: logs\2026-03-09\06-38-52
Successfully froze run artifacts to results\runs\baseline_2026-03-09_06-38-52
Val CER: None | Test CER: None



## 7) Notes

- This notebook assumes the repo already contains `config/model/bigru_ctc.yaml`.
- It keeps the same single-user subject workflow as the baseline runner.
- Run the smoke test first. If that succeeds, then run the full 40-epoch cell.
- After training, use the freeze cell so the run summary is copied into `results/`.
